# 07 Discussion

本 Notebook 為 NYCU R 語言課程期末專題的最終討論章節。

本章不重新訓練模型，也不改動前面 Notebook 的分析流程，而是整合：

- `03_Exploratory_Data_Analysis.ipynb`
- `04_Model_Development.ipynb`
- `05_Model_Evaluation.ipynb`
- `06_Model_Interpretation.ipynb`

並將本專題的重現結果與 Jain et al. (2024) 進行比較，最後說明再分析結果、限制、可重現性與學習反思。

## 7.1 Discussion Strategy

本章分成兩個主要部分：

1. **Reproduction Discussion**  
   比較本專題與 Jain et al. (2024) 在資料處理、模型建立、模型評估與圖表重現上的一致性。

2. **Reanalysis Discussion**  
   整理本專題額外完成的模型解釋、特徵重要性、permutation importance、residual analysis 與環境限制。

本章目標不是追求數值完全一致，而是確認分析流程是否合理、結果方向是否一致，並誠實說明造成差異的可能原因。

## 7.2 Environment Setup

In [ ]:
#--------------------------------------------------
# 7.2 Environment setup
#--------------------------------------------------

suppressPackageStartupMessages({
  library(tidyverse)
  library(data.table)
  library(knitr)
  library(scales)
})

#--------------------------------------------------
# Project paths
#--------------------------------------------------

project_root <- ".."
output_path <- file.path(project_root, "output")
figure_path <- file.path(project_root, "figures", "07_Discussion")

if (!dir.exists(output_path)) dir.create(output_path, recursive = TRUE)
if (!dir.exists(figure_path)) dir.create(figure_path, recursive = TRUE)

#--------------------------------------------------
# Table display option
#--------------------------------------------------

options(knitr.kable.NA = "")

cat("07 Discussion environment initialized.\n")

## 7.3 Load Available Outputs from Previous Notebooks

本節讀取前面 Notebook 匯出的結果檔案。若部分檔案不存在，Notebook 會保留說明，不會中斷執行。

In [ ]:
#--------------------------------------------------
# 7.3 Helper functions for loading outputs
#--------------------------------------------------

find_first_file <- function(pattern, path = output_path) {
  files <- list.files(path, pattern = pattern, recursive = TRUE, full.names = TRUE)
  if (length(files) == 0) return(NA_character_)
  return(files[1])
}

safe_read_csv <- function(file_path) {
  if (is.na(file_path) || !file.exists(file_path)) {
    return(NULL)
  }
  readr::read_csv(file_path, show_col_types = FALSE)
}

#--------------------------------------------------
# Search possible output files
#--------------------------------------------------

regression_metrics_file <- find_first_file("(?i)regression.*metrics.*\\.csv$")
classification_metrics_file <- find_first_file("(?i)classification.*metrics.*\\.csv$")
feature_importance_file <- find_first_file("(?i)feature.*importance.*\\.csv$")
permutation_importance_file <- find_first_file("(?i)permutation.*importance.*\\.csv$")

regression_metrics <- safe_read_csv(regression_metrics_file)
classification_metrics <- safe_read_csv(classification_metrics_file)
feature_importance_table <- safe_read_csv(feature_importance_file)
permutation_importance_table <- safe_read_csv(permutation_importance_file)

loaded_files <- tibble(
  Output = c(
    "Regression metrics",
    "Classification metrics",
    "Feature importance",
    "Permutation importance"
  ),
  File = c(
    regression_metrics_file,
    classification_metrics_file,
    feature_importance_file,
    permutation_importance_file
  ),
  Status = if_else(is.na(File), "Not found", "Loaded")
)

kable(loaded_files, caption = "Table 7.1 Loaded Output Files")

## 7.4 Reproduction Benchmark from Jain et al. (2024)

本節建立原論文主要模型表現的 benchmark，用於後續比較。

原論文在 regression evaluation 中主要使用 **R² score 與 p-value**；classification evaluation 則使用 **precision、recall、F1-score、support、AUC、Brier score 與 DeLong test**。本專題因 R 4.6.1 Apple Silicon 環境限制，CatBoost 與完整 SHAP 未執行，因此比較時會明確標示。

In [ ]:
#--------------------------------------------------
# 7.4.1 Regression benchmark from Jain et al. (2024)
#--------------------------------------------------

paper_regression_benchmark <- tribble(
  ~Dataset, ~Model, ~Paper_R2, ~Paper_p_value,
  "Non-newborn", "CatBoost Regression", 0.432, "<1e-2",
  "Non-newborn", "Random Forest Regression", 0.396, "<1e-2",
  "Non-newborn", "Linear Regression", 0.420, "<1e-2",
  "Newborn", "CatBoost Regression", 0.730, "<1e-2",
  "Newborn", "Random Forest Regression", 0.767, "<1e-2",
  "Newborn", "Linear Regression", 0.820, "<1e-2"
)

kable(
  paper_regression_benchmark,
  digits = 3,
  caption = "Table 7.2 Regression Benchmark Reported by Jain et al. (2024)"
)

In [ ]:
#--------------------------------------------------
# 7.4.2 Classification benchmark from Jain et al. (2024)
#--------------------------------------------------

paper_classification_benchmark <- tribble(
  ~Dataset, ~Model, ~Metric, ~Paper_Value,
  "Non-newborn", "Multinomial Logistic Regression", "Accuracy", 0.4698,
  "Newborn", "Random Forest Classifier", "Accuracy", 0.6008,
  "Non-newborn", "Logistic Regression", "Average AUC", 0.5522,
  "Non-newborn", "Random Forest Classifier", "Average AUC", 0.7800,
  "Non-newborn", "CatBoost Classifier", "Average AUC", 0.7844,
  "Newborn", "Logistic Regression", "Average AUC", 0.5220,
  "Newborn", "Random Forest Classifier", "Average AUC", 0.6798,
  "Newborn", "CatBoost Classifier", "Average AUC", 0.6962,
  "Non-newborn", "Logistic Regression", "Brier Score", 0.754,
  "Non-newborn", "Random Forest Classifier", "Brier Score", 0.644,
  "Non-newborn", "CatBoost Classifier", "Brier Score", 0.635,
  "Newborn", "Logistic Regression", "Brier Score", 0.780,
  "Newborn", "Random Forest Classifier", "Brier Score", 0.532,
  "Newborn", "CatBoost Classifier", "Brier Score", 0.635
)

kable(
  paper_classification_benchmark,
  digits = 4,
  caption = "Table 7.3 Classification Benchmark Reported by Jain et al. (2024)"
)

## 7.5 Reproduction Results from This Project

本節彙整本專題實際輸出的模型結果。若前面 Notebook 已成功輸出 metrics，會在此顯示；若尚未輸出，則以文字說明。

In [ ]:
#--------------------------------------------------
# 7.5.1 Display project regression metrics
#--------------------------------------------------

if (!is.null(regression_metrics)) {
  kable(
    regression_metrics,
    digits = 4,
    caption = "Table 7.4 Project Regression Metrics"
  )
} else {
  cat("Regression metrics file was not found. Please check output/Regression folder.\n")
}

In [ ]:
#--------------------------------------------------
# 7.5.2 Display project classification metrics
#--------------------------------------------------

if (!is.null(classification_metrics)) {
  kable(
    classification_metrics,
    digits = 4,
    caption = "Table 7.5 Project Classification Metrics"
  )
} else {
  cat("Classification metrics file was not found. Please check output/Classification folder.\n")
}

## 7.6 Comparison with Jain et al. (2024)

本專題的 reproduce 結果整體上與原論文流程高度一致，包含：

- 以 SPARCS 2016 inpatient discharge data 為資料來源。
- 移除 missing values 與 Unknown Admission。
- 將 newborn 與 non-newborn 分開建模。
- 移除與 Length of Stay 明顯相關、會造成資料洩漏的 cost / charge 欄位。
- Regression 與 classification 分開處理。
- Classification 使用五組 Length of Stay bins。
- 使用 linear regression、random forest、multinomial logistic regression 等可解釋模型。

主要差異來自執行環境與課程實作限制，而非研究設計差異。

In [ ]:
#--------------------------------------------------
# 7.6.1 Reproduction consistency table
#--------------------------------------------------

reproduction_consistency <- tribble(
  ~Component, ~Jain_2024, ~This_Project, ~Assessment,
  "Data source", "SPARCS 2016 inpatient discharge data", "SPARCS 2016 inpatient discharge data", "Consistent",
  "Sample size", "Approximately 2.3 million records after cleaning", "Approximately 2.3 million records after cleaning", "Consistent",
  "Newborn split", "Separate newborn and non-newborn models", "Separate newborn and non-newborn datasets", "Consistent",
  "Leakage variables", "Total Costs and Total Charges removed", "Total Costs and Total Charges removed", "Consistent",
  "Regression models", "Linear regression, random forest, CatBoost", "Linear regression, random forest; CatBoost retained but not run", "Partially consistent",
  "Classification models", "Multinomial logistic regression, random forest, CatBoost", "Multinomial logistic regression, random forest; CatBoost retained but not run", "Partially consistent",
  "Evaluation metrics", "R2, p-value, classifier metrics, AUC, Brier score", "R2, RMSE, MAE, accuracy, precision, recall, F1; AUC/Brier when probability available", "Partially consistent",
  "Interpretability", "SHAP and decision tree interpretation", "Feature importance, permutation importance, decision tree; SHAP not run", "Partially consistent",
  "Software", "Python-based pipeline", "R-based Jupyter Notebook pipeline", "Different implementation"
)

kable(
  reproduction_consistency,
  caption = "Table 7.6 Reproduction Consistency with Jain et al. (2024)"
)

### 7.6.2 Interpretation of Reproduction Differences

本專題與原論文的差異主要有四項：

1. **程式語言不同**  
   原論文使用 Python、Pandas、Scikit-learn、PyTorch 與 CatBoost；本專題依課程要求全程使用 R 與 Jupyter Notebook。

2. **CatBoost 未執行**  
   在 R 4.6.1 與 Apple Silicon 環境中，CatBoost R package 不易取得穩定 binary，因此保留流程但標示為 Not run。

3. **部分資料採樣**  
   Non-newborn 資料量超過 200 萬筆，若完整訓練所有模型會導致 Notebook 執行時間過長，因此在模型開發與解釋階段使用固定 random sampling。這會造成數值與原論文略有差異。

4. **SHAP 未完整重現**  
   原論文使用 SHAP 分析，但本專題在 R 環境下未安裝 `fastshap` 或 TreeSHAP 對應套件，因此改以 feature importance、permutation importance 與 decision tree 作為可解釋性替代方法。

## 7.7 Reanalysis Findings

本專題在完成 reproduction 後，額外進行 reanalysis。重點不是取代原論文結果，而是補充模型可解釋性與誤差理解。

In [ ]:
#--------------------------------------------------
# 7.7.1 Top feature importance summary
#--------------------------------------------------

if (!is.null(feature_importance_table)) {

  feature_col <- names(feature_importance_table)[str_detect(names(feature_importance_table), regex("feature", ignore_case = TRUE))][1]
  dataset_col <- names(feature_importance_table)[str_detect(names(feature_importance_table), regex("dataset", ignore_case = TRUE))][1]

  if (!is.na(feature_col) && !is.na(dataset_col)) {
    top_feature_summary <- feature_importance_table %>%
      group_by(.data[[dataset_col]]) %>%
      slice_head(n = 5) %>%
      ungroup()

    kable(
      top_feature_summary,
      digits = 4,
      caption = "Table 7.7 Top Features from Model Interpretation"
    )
  } else {
    cat("Feature importance table was loaded, but feature/dataset columns could not be identified.\n")
  }

} else {
  cat("Feature importance file was not found.\n")
}

In [ ]:
#--------------------------------------------------
# 7.7.2 Permutation importance summary
#--------------------------------------------------

if (!is.null(permutation_importance_table)) {

  kable(
    permutation_importance_table %>% arrange(desc(RMSE_Increase)) %>% slice_head(n = 15),
    digits = 4,
    caption = "Table 7.8 Top Permutation Importance Results"
  )

} else {
  cat("Permutation importance file was not found.\n")
}

### 7.7.3 Summary of Reanalysis Findings

綜合 06 的模型解釋結果，本專題得到以下觀察：

- 在 newborn model 中，**Birth Weight** 是最重要的預測因子之一，這與原論文指出 newborn LoS 可由 birth weight 解釋的結論一致。
- 在 non-newborn model 中，**APR DRG Code**、**APR Severity of Illness Code** 與 **Patient Disposition** 是較重要的特徵，與原論文對 non-newborn 模型的描述一致。
- Decision tree 提供了較容易被臨床與行政使用者理解的規則式模型，有助於解釋 LoS 預測。
- Permutation importance 補充了 impurity-based importance 的不足，可檢查特徵被打亂後模型誤差是否明顯增加。

## 7.8 Clinical and Health Services Implications

Length of Stay 預測具有明確的醫療管理意義。若醫院能在入院早期估計病人可能住院天數，可協助：

- 病床容量規劃。
- 住院成本與資源消耗估計。
- 高風險長住院族群辨識。
- 特定疾病分類或 DRG 群組的流程改善。
- 行政決策與醫療品質監測。

本專題結果顯示，使用開放式行政資料仍可建立具可解釋性的 LoS 預測模型。不過，此類資料缺乏生命徵象、檢驗數據、用藥資訊與共病細節，因此較適合用於 population-level 或 hospital-level 的管理分析，而非取代個別病人的臨床判斷。

## 7.9 Limitations

本專題限制如下：

1. **資料粒度限制**  
   SPARCS 屬於行政型資料，缺乏生理數據、實驗室檢查、用藥、護理評估與病程變化。

2. **無法追蹤同一病人的多次住院**  
   資料去識別化後無法分析 repeated admissions 或個人共病軌跡。

3. **CatBoost 未執行**  
   原論文最佳 non-newborn regression model 為 CatBoost regression，但本專題因 R 4.6.1 Apple Silicon 環境限制無法完整執行。

4. **SHAP 未完整重現**  
   原論文使用 SHAP 進行模型解釋；本專題以 feature importance、permutation importance 與 decision tree 替代。

5. **採樣造成數值差異**  
   為確保 Notebook 可在課程環境中順利執行，部分模型採固定 random sampling，可能造成 performance metrics 與原論文略有差異。

6. **分類機率輸出限制**  
   若部分模型未保存 probability matrix，則 AUC、Brier score 與 calibration plot 無法完整重現。

## 7.10 Reproducibility Reflection

本專題的主要收穫之一，是理解「可重現研究」不只是重新執行程式碼，而是需要清楚記錄：

- 資料來源。
- 前處理邏輯。
- 排除條件。
- 特徵選擇與移除理由。
- 模型訓練方式。
- 評估指標。
- 軟體版本與套件限制。
- 無法重現的原因。

本專題雖未能完全重現 CatBoost 與 SHAP，但已保留其方法位置與限制說明，並使用可執行的 R-based workflow 完成主要分析。這種誠實標示差異的方式，比強行產生不穩定或不可驗證的結果更符合 reproducible research 的精神。

## 7.11 AI Tool Use Record

本專題使用 AI 工具協助：

- 閱讀與整理 Jain et al. (2024) 的 Methods、Results 與 Evaluation。
- 協助規劃 Notebook 架構。
- 協助 R 程式除錯。
- 協助產生表格與圖形程式碼。
- 協助撰寫繁體中文 Markdown 說明。
- 協助整理 reproduction 與 reanalysis 的差異。

所有 AI 產出均由作者逐步執行、檢查錯誤訊息、修正程式碼並確認結果合理性後納入本專題。

## 7.12 Final Conclusion

本專題以 Jain et al. (2024) 為目標論文，使用 New York SPARCS 開放資料重現 hospital Length of Stay prediction 的主要流程。整體而言，本專題成功完成資料審查、前處理、EDA、模型開發、模型評估與模型解釋，並將 reproduction 與 reanalysis 清楚分開。

主要結論如下：

- 本專題成功重現原論文的資料處理邏輯與 newborn / non-newborn 分析設計。
- Regression 與 classification 模型流程與原論文高度一致。
- CatBoost 與 SHAP 因 R 4.6.1 Apple Silicon 環境限制未完整執行，但已誠實標示並以替代可解釋性方法補充。
- Reanalysis 顯示 newborn 的 Length of Stay 與 Birth Weight 高度相關，non-newborn 則主要與 APR DRG Code、APR Severity of Illness Code 與 Patient Disposition 相關。
- 本專題展現了使用 R 與 Jupyter Notebook 完成 SCI 論文數據重現與再分析的可行性。

## Notebook Execution Log

In [ ]:
#--------------------------------------------------
# 7.13 Final output log
#--------------------------------------------------

discussion_summary <- tibble(
  Item = c(
    "Reproduction discussion",
    "Reanalysis discussion",
    "Limitation summary",
    "Reproducibility reflection",
    "AI tool use record",
    "Final conclusion"
  ),
  Status = rep("Completed", 6)
)

write.csv(
  discussion_summary,
  file.path(output_path, "07_discussion_summary.csv"),
  row.names = FALSE
)

kable(
  discussion_summary,
  caption = "Table 7.9 Discussion Notebook Completion Summary"
)

cat("07 Discussion completed.\n")
cat("Output saved to:", file.path(output_path, "07_discussion_summary.csv"), "\n")